# EDA: синтетические банковские выписки МСП — кейс «Прогноз денежных потоков» (Центр-инвест, хакатон осень 2026)

**Зачем этот ноутбук.** Перед тем как строить категоризацию, прогноз остатка и детектор кассовых разрывов, нужно понять,
с какими данными мы работаем: что в них есть, насколько им можно доверять, какие закономерности заложил генератор и
какие ловушки нас ждут при валидации прогноза.

**Контекст кейса (кратко).** Сервис по истории операций прогнозирует остаток по дням минимум на 30 дней, находит будущие
кассовые разрывы (дата, сумма, вероятность) и объясняет их предпринимателю. Минимум: прототип, модуль категоризации,
модель прогноза с проверкой качества, интерфейс с графиком остатка и предупреждениями. Критерии: качество прогноза,
выявление разрывов (насколько заранее и корректно), доп. источники, техкачество и устойчивость к граничным случаям, презентация.

**Ответы менторов (чат, 23.10.2026):**
- ссылка на прототип до защиты — плюс, но демо обязательно; на защиту обычно 3–5 минут;
- ограничений на стек, ML-библиотеки и внешние API нет — выбор нужно аргументировать;
- для интеграций (1С, банк, реестры) **достаточно имитаций** (мок-адаптеров);
- звёздочки не обязательно делать все: баллы за каждую реализованную, приоритета нет, **но фича не засчитывается, если она не полностью рабочая**
  («функционал должен быть рабочим и готовым к использованию»); без полного выполнения минимальных требований звёздочки не засчитываются вовсе;
- **качество прогноза жюри проверяет на отдельном (скрытом) датасете** → решение должно обобщаться на новые компании, а не подгоняться под эти 150;
- интеграции: лучше отдельные сервисы, имитирующие стандартное поведение недоступных API (1С, ФНС, реестры); мок-заглушки — если не успеваете;
  главное — максимально реалистичное поведение конкретной интеграции. **«Просто всё на моках — нехорошо».**

**Структура:**
1. Загрузка и паспорт датасета
2. Качество данных: пропуски, дубли, сверка остатков, разрывы в выписке
3. Структура денежных потоков
4. Базовая категоризация операций (правила) и её покрытие
5. Остатки и кассовые разрывы: явные (минус на счёте) и скрытые (владелец докладывает деньги)
6. Регулярность и платёжный календарь (ЕНП, зарплата, аренда, кредиты)
7. Сезонность по отраслям
8. Контрагенты: концентрация, уход ключевого клиента, задержки оплат по счетам
9. Граничные случаи
10. Выводы и рекомендации для прогнозной модели и валидации
11. Экспорт HTML-отчёта

> Ноутбук рассчитан на Google Colab: при первом запуске он попросит загрузить `cashflow_dataset_participants.zip`
> (или положите распакованную папку рядом / укажите путь в `DATA_DIR`).

In [ ]:
import os, re, sys, glob, io, base64, zipfile, warnings, html as _html
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 120)

try:
    from IPython.display import display, Markdown
except ImportError:                       # запуск как обычного скрипта
    def display(x): print(x)
    def Markdown(x): return x

IN_COLAB = 'google.colab' in sys.modules
DATA_DIR = os.environ.get('CASHFLOW_DATA_DIR', '')   # можно вписать путь вручную, например '/content/drive/MyDrive/cashflow'

def find_data_root():
    for base in [DATA_DIR, '.', 'data', '/content', '/content/data']:
        if base and os.path.isdir(base):
            hits = glob.glob(os.path.join(base, '**', 'all_operations.csv'), recursive=True)
            if hits:
                return os.path.dirname(hits[0])
    return None

ROOT = find_data_root()
if ROOT is None:
    zips = glob.glob('*.zip') + glob.glob('/content/*.zip')
    if not zips and IN_COLAB:
        from google.colab import files
        zips = [k for k in files.upload() if k.endswith('.zip')]
    for z in zips:
        zipfile.ZipFile(z).extractall('data')
    ROOT = find_data_root()
assert ROOT, 'Не найден all_operations.csv: загрузите архив или задайте DATA_DIR'
print('Данные:', ROOT)

### Оформление графиков и сбор отчёта
Палитра — валидированная категориальная палитра (порядок слотов подобран под различимость при дальтонизме);
все графики и выводы параллельно складываются в `REPORT`, из которого в конце собирается HTML.

In [ ]:
C = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']   # категориальные слоты
CRIT, GOOD, WARN = '#d03b3b', '#0ca30c', '#fab219'                                            # статусы
INK, INK2, MUTED, GRID = '#0b0b0b', '#52514e', '#898781', '#e1e0d9'
SEQ = LinearSegmentedColormap.from_list('seq_blue', ['#f4f8fd', '#cde2fb', '#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b'])
DIV = LinearSegmentedColormap.from_list('div', ['#184f95', '#6da7ec', '#f0efec', '#ec8a89', '#b42f2f'])

plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK2, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'text.color': INK, 'axes.titlecolor': INK, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10, 'legend.frameon': False,
    'lines.linewidth': 2, 'figure.dpi': 110,
})

REPORT = []          # элементы отчёта: ('h', title, sub) | ('p', md) | ('finding', md) | ('fig', caption, b64) | ('table', caption, html) | ('kpi', list)

def rub(x, digits=1):
    """Короткая запись суммы: 1,2 млрд ₽ / 340 тыс. ₽."""
    a = abs(x)
    for div, unit in [(1e9, 'млрд'), (1e6, 'млн'), (1e3, 'тыс.')]:
        if a >= div:
            return f'{x/div:.{digits}f}'.replace('.', ',') + f' {unit} ₽'
    return f'{x:.0f} ₽'

def pct(x, digits=1):
    return f'{100*x:.{digits}f}%'.replace('.', ',')

def num(x):
    return f'{int(round(x)):,}'.replace(',', ' ')

def section(title, sub=''):
    REPORT.append(('h', title, sub))

def note(md):
    REPORT.append(('p', md))

def ru_dec(md):
    """3.1 -> 3,1 в тексте (даты вида 01.10.2024 не трогаем)."""
    return re.sub(r'(?<![.\d])(\d+)\.(\d{1,2})(?![.\d])', r'\1,\2', md)

def finding(md):
    md = ru_dec(md)
    display(Markdown('> **Вывод.** ' + md))
    REPORT.append(('finding', md))

def show_fig(fig, caption):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
    REPORT.append(('fig', caption, base64.b64encode(buf.getvalue()).decode()))
    plt.show()
    plt.close(fig)

def show_table(df, caption, index=False, max_rows=40):
    display(df.head(max_rows) if len(df) > max_rows else df)
    REPORT.append(('table', caption, df.head(max_rows).to_html(index=index, border=0, classes='tbl', na_rep='—', escape=True)))

## 1. Загрузка и паспорт датасета
Формат по README: UTF-8 с BOM, разделитель `;`, даты `ГГГГ-ММ-ДД`, суммы с точкой. **ИНН читаем строкой** — вымышленные ИНН
начинаются с `00`, при чтении числом нули теряются.

In [ ]:
ops = pd.read_csv(f'{ROOT}/all_operations.csv', sep=';', encoding='utf-8-sig',
                  dtype={'counterparty_inn': str, 'operation_id': str}, parse_dates=['operation_date'])
co = pd.read_csv(f'{ROOT}/companies.csv', sep=';', encoding='utf-8-sig',
                 dtype={'company_inn': str}, parse_dates=['first_operation_date', 'last_operation_date'])

SHORT = {
    'Общепит (кофейня, кафе)': 'Общепит', 'Оптовая торговля': 'Опт', 'Строительство и ремонт': 'Строительство',
    'Розница: продуктовый магазин': 'Продукты (розница)', 'IT-услуги и разработка': 'IT-услуги',
    'Продажи на маркетплейсах': 'Маркетплейсы', 'Грузоперевозки': 'Грузоперевозки', 'Автосервис': 'Автосервис',
    'Салон красоты': 'Салоны красоты', 'Производство мебели на заказ': 'Мебель на заказ',
    'Образовательный центр для детей': 'Детские центры', 'Розница: одежда и обувь': 'Одежда (розница)',
    'Сельское хозяйство (растениеводство)': 'Агро', 'Медицинская клиника / стоматология': 'Клиники',
    'Гостиница / база отдыха': 'Гостиницы',
}
co['ind'] = co.industry.map(SHORT).fillna(co.industry)
ops = ops.merge(co[['company_id', 'ind', 'legal_form']], on='company_id', how='left')
ops['inflow'] = ops.direction.eq('Поступление')
ops['signed'] = np.where(ops.inflow, ops.amount, -ops.amount)
ops['ym'] = ops.operation_date.dt.to_period('M')
ops['purpose'] = ops.payment_purpose.fillna('')

# сверка: отдельные выписки == общий файл
per_file = sum(len(pd.read_csv(f, sep=';', encoding='utf-8-sig', usecols=[0])) for f in glob.glob(f'{ROOT}/statements/*.csv'))
print(f'Строк в all_operations.csv: {len(ops):,}; суммарно в statements/*.csv: {per_file:,}; файлов: {len(glob.glob(f"{ROOT}/statements/*.csv"))}')
ops.head()

In [ ]:
def field_profile(df, cols):
    rows = []
    for c in cols:
        s = df[c]
        ex = s.dropna().astype(str)
        rows.append({'поле': c, 'тип': str(s.dtype), 'заполнено': pct(s.notna().mean() if c != 'payment_purpose'
                     else (df.purpose.str.strip() != '').mean()), 'уникальных': num(s.nunique()),
                     'пример': (ex.iloc[len(ex)//2][:70] if len(ex) else '')})
    return pd.DataFrame(rows)

cols = ['company_id', 'operation_id', 'operation_date', 'amount', 'direction', 'counterparty_inn', 'counterparty_name', 'payment_purpose', 'balance_eod']
prof = field_profile(ops, cols)

D0, D1 = ops.operation_date.min(), ops.operation_date.max()
inflow_sum, outflow_sum = ops.loc[ops.inflow, 'amount'].sum(), ops.loc[~ops.inflow, 'amount'].sum()
KPI = [
    ('Операций', num(len(ops))), ('Компаний', num(ops.company_id.nunique())),
    ('Период', f'{D0:%d.%m.%Y} – {D1:%d.%m.%Y}'), ('Отраслей', str(co.ind.nunique())),
    ('Поступления', rub(inflow_sum)), ('Списания', rub(outflow_sum)),
    ('Уникальных контрагентов', num(ops.counterparty_name.nunique())), ('ООО / ИП', f'{(co.legal_form=="ООО").sum()} / {(co.legal_form=="ИП").sum()}'),
]
REPORT.append(('kpi', KPI))
section('1. Паспорт датасета', 'Что за данные, какой период, кто в выборке')
show_table(prof, 'Профиль полей операции')
print(pd.DataFrame(KPI, columns=['метрика', 'значение']).to_string(index=False))

In [ ]:
cnt_check = ops.groupby('company_id').size().rename('факт').to_frame().join(co.set_index('company_id').operations_count)
mismatch_cnt = (cnt_check['факт'] != cnt_check['operations_count']).sum()

co['months'] = ((co.last_operation_date - co.first_operation_date).dt.days / 30.44).round(1)
co['lifecycle'] = np.select(
    [(co.first_operation_date <= D0 + pd.Timedelta(days=3)) & (co.last_operation_date >= D1 - pd.Timedelta(days=10)),
     co.last_operation_date < D1 - pd.Timedelta(days=10)],
    ['полная история', 'закрыла счёт / ушла'], 'открылась в периоде')
life = co.lifecycle.value_counts()
ops_month = ops.groupby(['company_id']).size().div(co.set_index('company_id').months.clip(lower=1)).rename('ops_per_month')
co = co.join(ops_month, on='company_id')

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={'width_ratios': [1.1, 1]})
order = co.ind.value_counts().index[::-1]
lf = pd.crosstab(co.ind, co.legal_form).reindex(order)
ax[0].barh(lf.index, lf['ООО'], color=C[0], label='ООО', height=0.7)
ax[0].barh(lf.index, lf['ИП'], left=lf['ООО'], color=C[1], label='ИП', height=0.7, edgecolor='#fcfcfb', linewidth=2)
ax[0].set_title('Компаний по отраслям'); ax[0].legend(loc='lower right'); ax[0].grid(axis='y', visible=False)
med = co.groupby('ind').ops_per_month.median().reindex(order)
ax[1].barh(med.index, med.values, color=C[0], height=0.7)
ax[1].set_title('Операций в месяц на компанию (медиана)'); ax[1].set_yticklabels([]); ax[1].grid(axis='y', visible=False)
for y, v in enumerate(med.values):
    ax[1].text(v, y, f' {v:.0f}', va='center', color=INK2, fontsize=9)
plt.tight_layout()
show_fig(fig, 'Состав выборки: 15 отраслей, ООО и ИП; интенсивность операций различается на порядок (розница с эквайрингом vs B2B)')

lc = co.groupby('lifecycle').agg(компаний=('company_id', 'size'), медиана_мес_истории=('months', 'median')).reset_index()
show_table(lc, 'Жизненный цикл компаний в выборке')
short_hist = co[co.months < 6].sort_values('months')[['company_id', 'company_name', 'ind', 'first_operation_date', 'last_operation_date', 'months', 'operations_count']]
show_table(short_hist, 'Компании с историей меньше 6 месяцев — «холодный старт» для прогноза')

finding(f'В выборке {co.company_id.nunique()} компаний, {num(len(ops))} операций за {D0:%d.%m.%Y}–{D1:%d.%m.%Y}. '
        f'Полная двухлетняя история у {life.get("полная история", 0)} компаний, {life.get("открылась в периоде", 0)} открылись по ходу, '
        f'{life.get("закрыла счёт / ушла", 0)} закрыли счёт. У {len(short_hist)} компаний история < 6 мес. — для них нужен '
        f'отраслевой fallback вместо индивидуальной модели. Интенсивность операций отличается на порядок: '
        f'от {med.min():.0f} до {med.max():.0f} операций в месяц (медианы по отраслям). '
        f'Справочник `companies.csv` согласован с операциями: расхождений в числе операций — {mismatch_cnt}.')

## 2. Качество данных
README честно предупреждает: пропуски дней/недель (остаток «скачет»), пустые назначения, пустые ИНН, единичные дубли,
возвраты ошибочных платежей. Проверяем каждое и — главное — **сверяем остаток**: `balance_eod(t) = balance_eod(t-1) + поступления(t) − списания(t)`.

In [ ]:
section('2. Качество данных', 'Пропуски, дубли, ИНН, сверка остатков, «дыры» в выписке')

# --- пропуски и ИНН
empty_purpose = ops.purpose.str.strip().eq('')
inn = ops.counterparty_inn
q_rows = [
    ('Пустое назначение платежа', empty_purpose.sum(), empty_purpose.mean()),
    ('Пустой ИНН контрагента', inn.isna().sum(), inn.isna().mean()),
    ('ИНН 10 знаков (организация)', inn.str.len().eq(10).sum(), inn.str.len().eq(10).mean()),
    ('ИНН 12 знаков (ИП / физлицо)', inn.str.len().eq(12).sum(), inn.str.len().eq(12).mean()),
    ('ИНН другой длины / нецифровой', (inn.notna() & ~inn.str.fullmatch(r'\d{10}|\d{12}', na=False)).sum(), 0),
    ('ИНН начинается с 00 (вымышленный)', inn.str.startswith('00', na=False).sum(), inn.str.startswith('00', na=False).mean()),
]
qtab = pd.DataFrame(q_rows, columns=['проверка', 'строк', 'доля'])
qtab['доля'] = qtab['доля'].map(pct); qtab['строк'] = qtab['строк'].map(num)
show_table(qtab, 'Пропуски и формат ИНН')

no_inn_by_cp = ops[inn.isna()].counterparty_name.str.replace(r'\s*\d+$', '', regex=True).value_counts().head(8).rename_axis('контрагент (без номера)').reset_index(name='строк без ИНН')
show_table(no_inn_by_cp, 'У кого не заполнен ИНН')

co_inn_ok = ((co.legal_form == 'ООО') & co.company_inn.str.len().eq(10)) | ((co.legal_form == 'ИП') & co.company_inn.str.len().eq(12))
print('ИНН компаний согласован с формой (ООО=10, ИП=12):', co_inn_ok.all())

In [ ]:
# --- дубли: одинаковые все поля, кроме operation_id
KEY = ['company_id', 'operation_date', 'amount', 'direction', 'counterparty_inn', 'counterparty_name', 'payment_purpose']
dup_extra = ops.duplicated(KEY, keep='first')
ops['is_dup'] = dup_extra

# --- сверка остатка по дням
day = (ops.groupby(['company_id', 'operation_date'])
          .agg(net=('signed', 'sum'), inflow=('amount', lambda s: s[ops.loc[s.index, 'inflow']].sum()),
               bal=('balance_eod', 'first'), n_bal=('balance_eod', 'nunique'), n=('amount', 'size'),
               dup_signed=('signed', lambda s: s[ops.loc[s.index, 'is_dup']].sum()))
          .reset_index())
day['outflow'] = day.inflow - day.net
day['prev_bal'] = day.groupby('company_id').bal.shift()
day['resid'] = (day.bal - day.prev_bal - day.net).round(2)

# Глобальный календарь рабочих дней = объединение дат всех 150 выписок (README: только рабочие дни по производственному календарю РФ)
CAL = pd.DatetimeIndex(sorted(ops.operation_date.unique()))
cal_idx = pd.Series(np.arange(len(CAL)), index=CAL)
day['wd_idx'] = day.operation_date.map(cal_idx)
day['missing_wd'] = day.groupby('company_id').wd_idx.diff() - 1          # пропущенных рабочих дней перед этой датой

mis = day[day.resid.abs() > 0.01].copy()
mis['причина'] = np.select([(mis.dup_signed != 0) & ((mis.resid + mis.dup_signed).abs() < 0.01),
                            mis.missing_wd >= 3],
                           ['дубль строки (в остатке не учтён)', 'пропуск в выписке (потеряны операции)'], 'прочее')
print('Дней с разными остатками внутри одного дня:', (day.n_bal > 1).sum())
print(f'Компания-дней: {len(day):,}; не сходится остаток: {len(mis)}')
recon = mis.groupby('причина').agg(дней=('resid', 'size'), компаний=('company_id', 'nunique'),
                                   медиана_скачка=('resid', lambda s: rub(s.abs().median())),
                                   макс_скачок=('resid', lambda s: rub(s.abs().max()))).reset_index()
show_table(recon, 'Сверка остатка: где balance_eod не равен предыдущему остатку + чистому потоку')

dup_tab = pd.DataFrame({'дублирующих строк': [dup_extra.sum()], 'компаний': [ops.loc[dup_extra, 'company_id'].nunique()],
                        'из них остаток не учитывает дубль': [(mis['причина'] == 'дубль строки (в остатке не учтён)').sum()]})
show_table(dup_tab, 'Дубли строк')

In [ ]:
# --- «дыры» в выписке
gaps = day[day.missing_wd >= 5][['company_id', 'operation_date', 'missing_wd', 'resid']].copy()
gaps['остаток_скачет'] = gaps.resid.abs() > 0.01
gap_summary = pd.DataFrame({
    'пропусков ≥5 раб. дней': [len(gaps)], 'компаний': [gaps.company_id.nunique()],
    'из них со скачком остатка': [gaps['остаток_скачет'].sum()],
    'макс. пропуск, раб. дней': [int(gaps.missing_wd.max()) if len(gaps) else 0]})
show_table(gap_summary, 'Пропуски в выписке (рабочие дни без единой операции)')

# пример: компания с самым большим скачком остатка
ex = mis.loc[mis.resid.abs().idxmax()]
g = day[day.company_id == ex.company_id]
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(g.operation_date, g.bal / 1e6, color=C[0], lw=1.6)
gap_start = CAL[int(ex.wd_idx - ex.missing_wd - 1)]
ax.axvspan(gap_start, ex.operation_date, color=WARN, alpha=0.25, lw=0)
ax.annotate(f'пропуск {int(ex.missing_wd)} раб. дн.\nскачок остатка {rub(ex.resid)}', xy=(ex.operation_date, ex.bal / 1e6),
            xytext=(20, 25), textcoords='offset points', fontsize=9, color=INK2, arrowprops=dict(arrowstyle='-', color=MUTED))
ax.axhline(0, color='#c3c2b7', lw=1)
ax.set_title(f'{ex.company_id}: остаток на конец дня, млн ₽'); ax.set_ylabel('млн ₽')
show_fig(fig, 'Пример «дыры» в выписке: операции за период не выгружены, остаток после пропуска не сходится с суммой потоков')

finding(f'Данные в целом чистые и внутренне согласованы: остаток одинаков для всех операций дня, и равенство '
        f'«вчерашний остаток + поступления − списания = сегодняшний остаток» выполняется в {pct(1 - len(mis)/day.prev_bal.notna().sum(), 2)} компания-дней. '
        f'Нарушения объясняются ровно двумя причинами: (1) {dup_extra.sum()} строк-дублей у {ops.loc[dup_extra,"company_id"].nunique()} компаний — '
        f'остаток их **не** учитывает, значит это артефакт выгрузки и их надо удалять; (2) пропуски выписки — '
        f'{len(gaps)} эпизодов ≥5 рабочих дней у {gaps.company_id.nunique()} компаний, после части из них остаток скачет на суммы до {rub(mis.resid.abs().max())}. '
        f'Следствие для модели: **целевой ряд — `balance_eod`, а не накопленная сумма потоков**; на «дырах» нельзя учить модель потоков '
        f'(нужно маскировать окна), а дубль определяется надёжно через сверку с остатком. Пустых назначений {pct(empty_purpose.mean(), 2)}, '
        f'пустых ИНН {pct(inn.isna().mean())} — в основном физлица и банк, поэтому ключ контрагента = ИНН, а при его отсутствии — наименование.')

# чистый датасет для дальнейшего анализа
ops = ops[~ops.is_dup].copy()

## 3. Структура денежных потоков

In [ ]:
section('3. Структура денежных потоков', 'Сколько, каких операций и какого размера')
st = ops.groupby('direction').agg(операций=('amount', 'size'), сумма=('amount', 'sum'), медиана=('amount', 'median'),
                                   среднее=('amount', 'mean'), p99=('amount', lambda s: s.quantile(.99)), максимум=('amount', 'max'))
st_disp = st.copy()
for c in ['сумма', 'медиана', 'среднее', 'p99', 'максимум']:
    st_disp[c] = st_disp[c].map(rub)
st_disp['операций'] = st_disp['операций'].map(num)
show_table(st_disp.reset_index(), 'Поступления vs списания')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
bins = np.linspace(0, 8, 65)
for i, (d, lab) in enumerate([('Поступление', 'Поступления'), ('Списание', 'Списания')]):
    ax[0].hist(np.log10(ops.loc[ops.direction == d, 'amount']), bins=bins, histtype='step', lw=2, color=C[i], label=lab)
ax[0].set_xticks(range(0, 9)); ax[0].set_xticklabels(['1', '10', '100', '1 тыс', '10 тыс', '100 тыс', '1 млн', '10 млн', '100 млн'], fontsize=8)
ax[0].set_title('Распределение сумм операций (лог-шкала)'); ax[0].legend(); ax[0].set_ylabel('операций')
srt = np.sort(ops.amount.values)[::-1]
cum = np.cumsum(srt) / srt.sum()
xs = np.arange(1, len(srt) + 1) / len(srt)
ax[1].plot(xs * 100, cum * 100, color=C[0])
ax[1].set_xscale('log'); ax[1].set_xlabel('топ-% операций по сумме'); ax[1].set_ylabel('% оборота')
top1 = cum[int(len(srt) * 0.01)]; top10 = cum[int(len(srt) * 0.10)]
ax[1].scatter([1, 10], [top1 * 100, top10 * 100], color=C[0], s=36, zorder=3)
ax[1].annotate(f'1% операций = {pct(top1, 0)} оборота', (1, top1 * 100), xytext=(8, -14), textcoords='offset points', fontsize=9, color=INK2)
ax[1].annotate(f'10% = {pct(top10, 0)}', (10, top10 * 100), xytext=(8, -14), textcoords='offset points', fontsize=9, color=INK2)
ax[1].set_title('Концентрация оборота')
plt.tight_layout()
show_fig(fig, 'Суммы распределены лог-нормально с тяжёлым хвостом: мелкие розничные зачисления по числу, крупные B2B-платежи по объёму')

finding(f'Поступлений по числу в {st.loc["Поступление","операций"]/st.loc["Списание","операций"]:.1f} раза больше, чем списаний '
        f'(ежедневные зачисления эквайринга и СБП), но медианное поступление {rub(st.loc["Поступление","медиана"])} против '
        f'{rub(st.loc["Списание","медиана"])} у списаний. Хвост очень тяжёлый: 1% самых крупных операций даёт {pct(top1,0)} оборота, '
        f'максимальная операция {rub(ops.amount.max())}. Для прогноза это значит: ошибка остатка определяется не тысячами мелких '
        f'зачислений, а несколькими крупными платежами — их нужно прогнозировать **поштучно** (календарь, счета, договоры), '
        f'а мелочь — агрегировано (дневной ряд выручки). Метрики считать в рублях и относительно масштаба компании (MASE / ошибка в % от среднего оборота).')

## 4. Базовая категоризация операций
Назначения платежей — шаблонные тексты с вариативностью формулировок («Единый налоговый платеж» / «ЕНП» / «Единый налоговый платёж»,
три варианта записи НДС и т.д.). Ниже — прозрачный бейзлайн на регулярных выражениях; пустые назначения доразмечаются
по истории того же контрагента. Это и есть первый кандидат в «модуль категоризации» прототипа.

In [ ]:
RULES_IN = [
    ('Эквайринг (карты)',            r'эквайринг'),
    ('СБП (QR)',                     r'\bСБП\b'),
    ('Наличная выручка',             r'взнос наличных|инкасс'),
    ('Маркетплейсы: выплаты',        r'договору? оферты.*за период|маркетплейс'),
    ('Займ / пополнение от владельца', r'займ|задолженности по овердрафту|уставн'),
    ('Кредит получен',               r'выдач[аи] кредит|предоставлени[ея] кредит|зачислени[ея] кредит|транш'),
    ('Перевод собственных средств',  r'собственн|с расчетного счета в'),
    ('Возврат / ошибочный платёж',   r'возврат|ошибочн'),
    ('Субсидии, страховка',          r'субсид|грант|страхов'),
    ('Оплата от клиентов',           r'оплат|предоплат|расчет|аванс|доплат|агентск|по договору|по счету|упд'),
]
RULES_OUT = [
    ('Налоги (ЕНП)',                 r'единый налог|\bЕНП\b|налогов|страховые взносы|пени|штраф'),
    ('Взыскания (исп. лист)',        r'исполнительн'),
    ('Зарплата',                     r'заработн|зарплат|отпускн|больничн'),
    ('ГПХ / самозанятые',            r'ГПХ|самозанят'),
    ('Вывод владельцу / дивиденды',  r'дивиденд|личн|на счет физического лица|займа учредител|подотчет'),
    ('Банк: комиссии',               r'комисси|POS-терминал|ведени[ея] счета|расчетно-кассов'),
    ('Кредиты и проценты',           r'кредит|овердрафт|процент'),
    ('Лизинг',                       r'лизинг'),
    ('Аренда',                       r'аренд'),
    ('Коммунальные услуги',          r'электроэнерг|водоснабж|водоотвед|ТКО|теплоснабж|отоплен|газоснабж'),
    ('Перевод собственных средств',  r'собственн'),
    ('Возврат / ошибочный платёж',   r'возврат|ошибочн'),
    ('Реклама и маркетинг',          r'реклам|продвижен'),
    ('Связь, ПО, бухгалтерия',       r'связи|интернет|(?-i:\bПО\b)|программ|бухгалт|(?-i:1С)|подписк|облачн'),
    ('Логистика и топливо',          r'топлив|провоз|транспорт|экспедиц|ГСМ|бензин|перевозк'),
    ('Хоз. услуги',                  r'охран|уборк|клининг|канцтовар|хозтовар'),
    ('Ремонт и оборудование',        r'ремонт|оборудован|монтаж'),
    ('Закупки у поставщиков',        r'УПД|поставк|товар|продукт|продукц|материал|сырь|запчаст|оплата по счету|субподряд|изделия'),
]
_RIN = [(c, re.compile(p, re.I)) for c, p in RULES_IN]
_ROUT = [(c, re.compile(p, re.I)) for c, p in RULES_OUT]

def categorize(direction, text):
    if not text.strip():
        return None
    for c, rx in (_RIN if direction == 'Поступление' else _ROUT):
        if rx.search(text):
            return c
    return 'Прочее'

ops['category'] = [categorize(d, t) for d, t in zip(ops.direction, ops.purpose)]
ops['cat_source'] = np.where(ops.category.notna(), 'назначение', None)

# пустые назначения: самая частая категория у того же контрагента в той же компании и направлении
ops['cp_key'] = ops.counterparty_inn.fillna(ops.counterparty_name)
mode_cat = (ops[ops.category.notna()].groupby(['company_id', 'direction', 'cp_key']).category
               .agg(lambda s: s.value_counts().index[0]))
fill = ops.loc[ops.category.isna()].join(mode_cat.rename('cat_fill'), on=['company_id', 'direction', 'cp_key']).cat_fill
ops.loc[fill.index, 'category'] = fill
ops.loc[fill.dropna().index, 'cat_source'] = 'история контрагента'
ops['category'] = ops.category.fillna('Не определено')
ops.loc[ops.category.eq('Не определено'), 'cat_source'] = 'нет данных'

OPERATING_IN = ['Эквайринг (карты)', 'СБП (QR)', 'Наличная выручка', 'Маркетплейсы: выплаты', 'Оплата от клиентов']
OWNER_IN = ['Займ / пополнение от владельца', 'Перевод собственных средств']
ops['is_revenue'] = ops.inflow & ops.category.isin(OPERATING_IN)

cat_tab = (ops.groupby(['direction', 'category']).agg(операций=('amount', 'size'), сумма=('amount', 'sum'), компаний=('company_id', 'nunique'))
              .reset_index().sort_values(['direction', 'сумма'], ascending=[True, False]))
cat_tab['доля суммы'] = cat_tab.groupby('direction').сумма.transform(lambda s: s / s.sum()).map(pct)
cat_tab['сумма'] = cat_tab['сумма'].map(rub); cat_tab['операций'] = cat_tab['операций'].map(num)
section('4. Категоризация операций', 'Прозрачный бейзлайн на правилах + доразметка по истории контрагента')
show_table(cat_tab, 'Категории операций (после правил и доразметки пустых назначений)', max_rows=60)

unk = ops.category.isin(['Прочее', 'Не определено'])
src = ops.cat_source.value_counts()
print(ops.loc[ops.category == 'Прочее', 'purpose'].str.replace(r'\d+', '#', regex=True).str[:90].value_counts().head(10))

# варианты формулировок налогового платежа
tax_variants = ops[ops.category == 'Налоги (ЕНП)'].purpose.str.strip().str.replace(r'\d+', '#', regex=True).value_counts().head(8)
show_table(tax_variants.rename_axis('назначение').reset_index(name='операций'), 'Одна сущность — разные формулировки: налоговый платёж')

In [ ]:
def share_heatmap(direction, title, ax):
    sub = ops[ops.direction == direction]
    m = sub.pivot_table(index='ind', columns='category', values='amount', aggfunc='sum', fill_value=0)
    m = m.div(m.sum(axis=1), axis=0)
    m = m[m.sum().sort_values(ascending=False).index]
    m = m.loc[:, m.max() >= 0.02]
    im = ax.imshow(m.values, cmap=SEQ, vmin=0, vmax=min(1, m.values.max()), aspect='auto')
    ax.set_xticks(range(m.shape[1])); ax.set_xticklabels(m.columns, rotation=40, ha='right', fontsize=8.5)
    ax.set_yticks(range(m.shape[0])); ax.set_yticklabels(m.index, fontsize=9)
    ax.grid(False)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v = m.values[i, j]
            if v >= 0.05:
                ax.text(j, i, f'{v*100:.0f}', ha='center', va='center', fontsize=7.5, color='white' if v > 0.45 else INK)
    ax.set_title(title)
    return m

fig, ax = plt.subplots(1, 2, figsize=(15, 6.2))
m_in = share_heatmap('Поступление', 'Структура поступлений, % суммы', ax[0])
m_out = share_heatmap('Списание', 'Структура списаний, % суммы', ax[1])
ax[1].set_yticklabels([])
plt.tight_layout()
show_fig(fig, 'Отраслевые «отпечатки» денежного потока: розница живёт эквайрингом и СБП, B2B — оплатами по счетам; в списаниях доминируют закупки, зарплата и налоги')

finding(f'Простые правила покрывают {pct(1 - unk.mean(), 2)} операций по числу и {pct(1 - ops.loc[unk,"amount"].sum()/ops.amount.sum(), 2)} по сумме; '
        f'пустые назначения ({num((ops.cat_source != "назначение").sum())} строк) на {pct(src.get("история контрагента",0)/max(1,(ops.cat_source != "назначение").sum()),0)} '
        f'доразмечены по истории того же контрагента. Генератор использует конечный набор шаблонов с вариациями записи — '
        f'поэтому **правила + словарь контрагентов дают почти идеальное качество**, и тяжёлая ML/LLM-категоризация не нужна '
        f'(а с учётом 152-ФЗ и нежелательна). ML-классификатор (TF-IDF + логрег) уместен как второй слой для «Прочего» '
        f'и для демонстрации сравнения подходов. Важно: контрагент «ПАО «Банк»» — это и эквайринг, и СБП, и комиссии, и кредит, '
        f'т.е. категория определяется текстом, а не контрагентом. Отраслевые профили потока сильно различаются — это аргумент '
        f'за отраслевые априорные шаблоны для компаний с короткой историей.')

## 5. Остатки и кассовые разрывы
Строим непрерывный дневной ряд остатка на рабочих днях (дни без операций — остаток предыдущего дня).
**Явный разрыв** — остаток ниже нуля (овердрафт). **Скрытый разрыв** — день, когда без поступления денег от владельца
(займ, перевод с другого счёта) остаток ушёл бы в минус: README прямо говорит, что генератор моделирует «поведение владельца, когда денег не хватает».

In [ ]:
section('5. Остатки и кассовые разрывы', 'Явные (минус на счёте) и скрытые (владелец докладывает деньги)')

# дневная панель остатков
day = (ops.groupby(['company_id', 'operation_date'])
          .agg(net=('signed', 'sum'), bal=('balance_eod', 'first'),
               revenue=('amount', lambda s: s[ops.loc[s.index, 'is_revenue']].sum()),
               owner_in=('amount', lambda s: s[ops.loc[s.index, 'inflow'] & ops.loc[s.index, 'category'].isin(OWNER_IN)].sum()),
               outflow=('amount', lambda s: s[~ops.loc[s.index, 'inflow']].sum()))
          .reset_index())
parts = []
for cid, g in day.groupby('company_id'):
    dates = CAL[(CAL >= g.operation_date.min()) & (CAL <= g.operation_date.max())]
    p = g.set_index('operation_date').reindex(dates)
    p['bal'] = p.bal.ffill()
    p[['net', 'revenue', 'owner_in', 'outflow']] = p[['net', 'revenue', 'owner_in', 'outflow']].fillna(0)
    p['company_id'] = cid
    parts.append(p.rename_axis('date').reset_index())
panel = pd.concat(parts, ignore_index=True)
panel = panel.merge(co[['company_id', 'ind']], on='company_id')

# масштаб: средний дневной расход компании -> «дней покрытия» = остаток / средний дневной расход
avg_out = panel.groupby('company_id').outflow.mean().rename('avg_out')
panel = panel.join(avg_out, on='company_id')
panel['cover_days'] = panel.bal / panel.avg_out.replace(0, np.nan)

# явные разрывы: эпизоды отрицательного остатка
panel['neg'] = panel.bal < 0
panel['run'] = (panel.neg != panel.groupby('company_id').neg.shift()).cumsum()
ep = (panel[panel.neg].groupby(['company_id', 'run'])
          .agg(начало=('date', 'min'), конец=('date', 'max'), раб_дней=('date', 'size'), мин_остаток=('bal', 'min'), ind=('ind', 'first'))
          .reset_index(level=1, drop=True).reset_index())

# скрытые разрывы: без денег владельца остаток в этот день был бы < 0
panel['hidden_gap'] = (panel.owner_in > 0) & (panel.bal - panel.owner_in < 0) & (panel.bal >= 0)
hid = panel[panel.hidden_gap]

neg_cos = ep.company_id.nunique()
ovd_cos = ops.loc[ops.purpose.str.contains('овердрафт', case=False), 'company_id'].nunique()
ep_disp = ep.sort_values('мин_остаток').head(15).copy()
ep_disp['мин_остаток'] = ep_disp['мин_остаток'].map(rub)
show_table(ep_disp, 'Самые глубокие эпизоды отрицательного остатка (явные разрывы)')

gap_by_ind = (pd.DataFrame({'компаний': co.groupby('ind').size(),
                            'с минусом на счёте': ep.groupby('ind').company_id.nunique(),
                            'со скрытым разрывом': hid.groupby('ind').company_id.nunique()})
                .fillna(0).astype(int).sort_values('с минусом на счёте', ascending=False))
show_table(gap_by_ind.reset_index().rename(columns={'ind': 'отрасль'}), 'Разрывы по отраслям')

In [ ]:
# поведение владельца: в каком состоянии счёта приходят деньги владельца
own = ops[ops.inflow & ops.category.isin(OWNER_IN)][['company_id', 'operation_date', 'amount', 'category']]
pv = panel[['company_id', 'date', 'bal', 'cover_days']].copy()
pv['prev_cover'] = pv.groupby('company_id').cover_days.shift()
own = own.merge(pv.rename(columns={'date': 'operation_date'}), on=['company_id', 'operation_date'], how='left')
own['bal_before'] = own.bal - own.amount       # грубо: остаток дня без этого поступления
med_cover_all = panel.cover_days.median()
med_cover_before = (own.bal_before / own.merge(avg_out, left_on='company_id', right_index=True).avg_out).median()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1.25, 1]})
ind_order = panel.groupby('ind').cover_days.median().sort_values().index
data = [panel.loc[panel.ind == i, 'cover_days'].clip(-20, 150).dropna().values for i in ind_order]
bp = ax[0].boxplot(data, vert=False, labels=ind_order, showfliers=False, patch_artist=True, widths=0.6,
                   medianprops=dict(color=INK, lw=1.5), boxprops=dict(facecolor='#cde2fb', edgecolor=C[0]),
                   whiskerprops=dict(color=C[0]), capprops=dict(color=C[0]))
ax[0].axvline(0, color=CRIT, lw=1.2); ax[0].set_xlabel('дней покрытия = остаток / средний дневной расход')
ax[0].set_title('Запас денег по отраслям'); ax[0].grid(axis='y', visible=False)
x = (own.bal_before / own.merge(avg_out, left_on='company_id', right_index=True).avg_out).clip(-30, 120)
ax[1].hist(panel.cover_days.clip(-30, 120).dropna(), bins=60, density=True, histtype='step', lw=2, color=C[0], label='все дни')
ax[1].hist(x.dropna(), bins=60, density=True, histtype='step', lw=2, color=C[1], label='остаток без денег владельца\nв день их поступления')
ax[1].axvline(0, color=CRIT, lw=1.2); ax[1].legend(fontsize=8.5); ax[1].set_title('Когда владелец докладывает деньги')
ax[1].set_xlabel('дней покрытия'); ax[1].set_yticks([])
plt.tight_layout()
show_fig(fig, 'Деньги владельца приходят именно тогда, когда остаток близок к нулю или ушёл бы в минус — это «замаскированные» разрывы')

# примеры рядов остатка
top_neg = panel.groupby('company_id').neg.sum().sort_values(ascending=False).head(6).index
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True)
for a, cid in zip(axes.flat, top_neg):
    g = panel[panel.company_id == cid]
    a.plot(g.date, g.bal / 1e6, color=C[0], lw=1.2)
    a.fill_between(g.date, g.bal / 1e6, 0, where=g.bal < 0, color=CRIT, alpha=0.35, lw=0)
    h = g[g.hidden_gap]
    a.scatter(h.date, h.bal / 1e6, s=14, color=C[1], zorder=3)
    a.axhline(0, color='#c3c2b7', lw=1)
    a.set_title(f'{cid} · {g.ind.iloc[0]}', fontsize=10)
    a.tick_params(axis='x', labelsize=8, rotation=30)
fig.supylabel('остаток, млн ₽', color=INK2, fontsize=10)
plt.tight_layout()
show_fig(fig, 'Компании с самым долгим отрицательным остатком. Красная заливка — минус на счёте, оранжевые точки — дни, когда владелец закрыл бы минус своими деньгами')

HOLD0 = pd.Timestamp('2026-07-01')
hold_neg = ep[ep.конец >= HOLD0].company_id.nunique()
hold_hid = hid[hid.date >= HOLD0].company_id.nunique()
hold_any = pd.concat([ep[ep.конец >= HOLD0].company_id, hid[hid.date >= HOLD0].company_id]).nunique()

finding(f'Явный разрыв (минус на счёте) хотя бы раз был у {neg_cos} из {co.company_id.nunique()} компаний '
        f'({len(ep)} эпизодов, медиана {ep.раб_дней.median():.0f} раб. дн., самый глубокий {rub(ep.мин_остаток.min())}); '
        f'платежи за овердрафт видны у {ovd_cos} компаний — это подтверждает, что минус — это реальный овердрафт, а не ошибка данных. '
        f'Но **ещё {hid.company_id.nunique()} компаний имеют скрытые разрывы**: в {len(hid)} дней остаток ушёл бы в минус, '
        f'если бы владелец не довнёс деньги (займ, перевод со своего счёта в другом банке). Медианный запас перед таким поступлением — '
        f'{med_cover_before:.1f} дн. расхода против {med_cover_all:.0f} дн. в обычный день. '
        f'**Вывод для разметки:** «разрыв» нужно определять не по факту `balance_eod < 0`, а по прогнозу остатка **без** будущих денег '
        f'владельца (они — реакция на разрыв, а не плановый поток), и сравнивать с порогом «неснижаемого остатка» (например, ближайшие '
        f'обязательные платежи). В отложенной выборке (июль–сентябрь 2026) разрывы есть у {hold_any} компаний '
        f'({hold_neg} явные, {hold_hid} скрытые) — на них и надо считать recall / заблаговременность / ложные тревоги.')

## 6. Регулярность и платёжный календарь
Прогноз держится на «скелете» регулярных платежей. Смотрим, в какие дни недели/месяца они происходят и какая доля расходов предсказуема.

In [ ]:
section('6. Регулярность и платёжный календарь', 'Дни недели, дни месяца, периодические платежи')
ops['wd'] = ops.operation_date.dt.dayofweek
wd_names = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб']
retail = ops[ops.category.isin(['Эквайринг (карты)', 'СБП (QR)'])]
n_days_by_wd = pd.Series(CAL.dayofweek).value_counts().reindex(range(6)).fillna(0)
r_wd = retail.groupby('wd').amount.sum().reindex(range(6)).fillna(0) / n_days_by_wd.replace(0, np.nan)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].bar(wd_names, (r_wd / r_wd.mean()).values, color=C[0], width=0.6)
ax[0].axhline(1, color=MUTED, lw=1, ls='--'); ax[0].set_title('Зачисления эквайринга+СБП\nв среднем за рабочий день (индекс)')
dom = lambda cats: ops[ops.category.isin(cats) & ~ops.inflow].operation_date.dt.day
ax[1].hist(dom(['Налоги (ЕНП)']), bins=np.arange(0.5, 32), color=C[0], rwidth=0.8)
ax[1].set_title('ЕНП: день месяца'); ax[1].set_xlabel('число')
sal = ops[ops.category.eq('Зарплата')]
is_adv = sal.purpose.str.contains('аванс', case=False)
ax[2].hist([sal[is_adv].operation_date.dt.day, sal[~is_adv].operation_date.dt.day], bins=np.arange(0.5, 32), stacked=True,
           color=[C[0], C[1]], label=['аванс', 'окончательный расчёт'], rwidth=0.8)
ax[2].legend(fontsize=8.5); ax[2].set_title('Зарплата: день месяца'); ax[2].set_xlabel('число')
plt.tight_layout()
show_fig(fig, 'Календарные эффекты: в понедельник приходят деньги за выходные; ЕНП — у 28-го числа (сдвиг на рабочий день); зарплата — две волны')

tax_m = ops[ops.category.eq('Налоги (ЕНП)')].groupby('ym').amount.sum()
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(tax_m.index.to_timestamp(), tax_m.values / 1e6, color=C[0], marker='o', ms=4)
for t, v in tax_m.items():
    if t.month in (4, 7, 10, 1) or v == tax_m.max():
        ax.annotate(f'{t.month:02d}.{t.year % 100}', (t.to_timestamp(), v / 1e6), xytext=(0, 6), textcoords='offset points', ha='center', fontsize=8, color=INK2)
ax.set_title('Налоговые платежи всех компаний по месяцам, млн ₽'); ax.set_ylabel('млн ₽')
show_fig(fig, 'Налоги имеют квартальный ритм (авансы УСН и НДС после квартала) и годовой пик — налоговый календарь даёт прогнозу «бесплатные» точные события')

In [ ]:
# детектор периодических платежей: компания × контрагент × категория
def periodicity(g):
    d = g.operation_date.sort_values().diff().dt.days.dropna()
    if len(d) < 3:
        return None
    med = d.median(); cv = d.std() / med if med else np.nan
    amt_cv = g.amount.std() / g.amount.mean()
    kind = ('еженедельно' if 5 <= med <= 9 else 'раз в 2 недели' if 12 <= med <= 17 else 'ежемесячно' if 26 <= med <= 35
            else 'ежеквартально' if 80 <= med <= 100 else None)
    if kind is None or cv > 0.35:
        return None
    return pd.Series({'период': kind, 'медиана_интервала': med, 'cv_интервала': cv, 'cv_суммы': amt_cv, 'платежей': len(g), 'сумма': g.amount.sum()})

out = ops[~ops.inflow]
rec = out.groupby(['company_id', 'cp_key', 'category']).apply(periodicity).dropna(how='all')
rec = rec.reset_index() if len(rec) else pd.DataFrame()
rec_share_cat = (rec.groupby('category').сумма.sum() / out.groupby('category').amount.sum()).dropna().sort_values(ascending=False)
rec_share_co = (rec.groupby('company_id').сумма.sum() / out.groupby('company_id').amount.sum()).reindex(co.company_id).fillna(0)
rec_share_ind = rec_share_co.groupby(co.set_index('company_id').ind).median().sort_values()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
ax[0].barh(rec_share_cat.index[::-1], rec_share_cat.values[::-1] * 100, color=C[0], height=0.65)
ax[0].set_title('Доля регулярных платежей в категории, %'); ax[0].grid(axis='y', visible=False)
ax[1].barh(rec_share_ind.index, rec_share_ind.values * 100, color=C[0], height=0.65)
ax[1].set_title('Доля регулярных списаний у компании, % (медиана)'); ax[1].grid(axis='y', visible=False)
plt.tight_layout()
show_fig(fig, 'Регулярные (периодические по контрагенту) списания: аренда, связь, ПО, коммуналка — почти полностью; закупки — частично')

rec_kinds = rec.groupby('период').agg(серий=('сумма', 'size'), сумма=('сумма', 'sum')).reset_index()
rec_kinds['сумма'] = rec_kinds.сумма.map(rub)
show_table(rec_kinds, 'Найдено регулярных серий списаний (компания × контрагент × категория)')

finding(f'Календарь в данных выражен очень чётко и легко извлекается: зачисления эквайринга в понедельник в '
        f'{r_wd.iloc[0]/r_wd.iloc[1:5].mean():.1f} раза выше среднего вт–пт (выручка выходных зачисляется в следующий рабочий день); '
        f'ЕНП платится 28-го или в ближайший рабочий день; зарплата — двумя волнами (аванс и окончательный расчёт). '
        f'Простой детектор периодичности находит {len(rec)} регулярных серий списаний; в медиане у компании '
        f'{pct(rec_share_co.median(),0)} суммы списаний — регулярные платежи. Это основа прогноза: «скелет» из известных платежей '
        f'(аренда, кредит, лизинг, ЕНП, зарплата, подписки) + статистическая модель для нерегулярной части (выручка, закупки).')

## 7. Сезонность по отраслям
Индекс месяца = операционная выручка компании в месяце / её средняя месячная выручка (без неполных первого и последнего месяцев), затем среднее по отрасли.

In [ ]:
section('7. Сезонность', 'Индекс месячной выручки по отраслям')
rev_m = ops[ops.is_revenue].groupby(['company_id', 'ym']).amount.sum().unstack(fill_value=0).stack().rename('rev').reset_index()
bounds = ops.groupby('company_id').ym.agg(['min', 'max'])
rev_m = rev_m.join(bounds, on='company_id')
rev_m = rev_m[(rev_m.ym > rev_m['min']) & (rev_m.ym < rev_m['max'])]
rev_m['idx'] = rev_m.rev / rev_m.groupby('company_id').rev.transform('mean')
rev_m = rev_m.merge(co[['company_id', 'ind']], on='company_id')
rev_m['month'] = rev_m.ym.dt.month
seas = rev_m.groupby(['ind', 'month']).idx.mean().unstack()
amp = (seas.max(axis=1) - seas.min(axis=1)).sort_values(ascending=False)
seas = seas.loc[amp.index]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(seas.values, cmap=DIV, norm=TwoSlopeNorm(vcenter=1, vmin=min(0.2, seas.values.min()), vmax=max(1.8, seas.values.max())), aspect='auto')
mn = ['янв', 'фев', 'мар', 'апр', 'май', 'июн', 'июл', 'авг', 'сен', 'окт', 'ноя', 'дек']
ax.set_xticks(range(12)); ax.set_xticklabels(mn); ax.set_yticks(range(len(seas))); ax.set_yticklabels(seas.index); ax.grid(False)
for i in range(seas.shape[0]):
    for j in range(12):
        v = seas.values[i, j]
        ax.text(j, i, f'{v:.2f}'.replace('.', ','), ha='center', va='center', fontsize=7.5, color='white' if abs(v - 1) > 0.55 else INK)
cb = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02); cb.set_label('индекс (1 = средний месяц)', color=INK2)
ax.set_title('Сезонность выручки по отраслям (отсортировано по амплитуде)')
show_fig(fig, 'Сезонность: курорт — лето, детские центры — сентябрь–май, агро — урожай осенью, розница — декабрь')

finding(f'Сезонность сильная и отраслевая: максимальная амплитуда у «{amp.index[0]}» (от {seas.iloc[0].min():.2f} до {seas.iloc[0].max():.2f} среднего месяца), '
        f'затем {", ".join(amp.index[1:4])}; самые ровные — {", ".join(amp.index[-3:])}. При горизонте 30 дней и истории 1,5–2 года '
        f'годовой сезонный профиль оценивается всего по 1–2 наблюдениям на месяц — поэтому для коротких историй лучше брать '
        f'**отраслевой** профиль сезонности, а валидироваться обязательно на нескольких точках отсечки (rolling origin), '
        f'иначе одна отсечка в «тихом» сезоне даст обманчиво хорошую метрику.')

## 8. Контрагенты: концентрация, уход клиента, задержки оплат

In [ ]:
section('8. Контрагенты', 'Концентрация клиентов, уход ключевого клиента, задержки оплат по счетам')
b2b = ops[ops.inflow & ops.category.eq('Оплата от клиентов')]
cli = b2b.groupby(['company_id', 'cp_key']).amount.sum()
conc = cli.groupby('company_id').agg(top1=lambda s: s.max() / s.sum(), clients='size', hhi=lambda s: ((s / s.sum()) ** 2).sum())
b2b_share = (b2b.groupby('company_id').amount.sum() / ops[ops.is_revenue].groupby('company_id').amount.sum()).rename('b2b_share')
conc = conc.join(b2b_share).join(co.set_index('company_id').ind)
conc_b2b = conc[conc.b2b_share > 0.5]

# уход ключевого клиента: доля ≥15% и платежей не было последние 120 дней при живой компании
last_pay = b2b.groupby(['company_id', 'cp_key']).operation_date.max()
co_last = co.set_index('company_id').last_operation_date
key = (cli / cli.groupby('company_id').transform('sum')).rename('share').to_frame().join(last_pay.rename('last_pay'))
key = key[key.index.get_level_values(0).isin(conc[(conc.clients >= 3) & (conc.ind != 'Маркетплейсы')].index)]
key['co_last'] = key.index.get_level_values(0).map(co_last)
lost = key[(key.share >= 0.15) & ((key.co_last - key.last_pay).dt.days > 120)].reset_index()

# задержка оплаты: дата документа из назначения («по УПД № 279 от 19.08.2024») -> дата зачисления
doc_date = pd.to_datetime(b2b.purpose.str.extract(r'от (\d{2}\.\d{2}\.\d{4})')[0], format='%d.%m.%Y', errors='coerce')
lag = (b2b.operation_date - doc_date).dt.days
b2b = b2b.assign(lag=lag)
lag_ok = b2b[(b2b.lag >= 0) & (b2b.lag <= 365)]
lag_ind = lag_ok.groupby('ind').lag.describe(percentiles=[.5, .9])[['count', '50%', '90%']].sort_values('50%')
partial = b2b.purpose.str.contains('Частичная оплата|Доплата', case=False).mean()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
d = conc_b2b.sort_values('top1')
ax[0].hist(conc_b2b.top1 * 100, bins=np.arange(0, 105, 5), color=C[0], rwidth=0.85)
ax[0].set_title('Доля крупнейшего клиента в B2B-выручке, %'); ax[0].set_xlabel('%'); ax[0].set_ylabel('компаний')
ax[1].barh(lag_ind.index, lag_ind['90%'], color='#cde2fb', height=0.65, label='90-й перцентиль')
ax[1].barh(lag_ind.index, lag_ind['50%'], color=C[0], height=0.65, label='медиана')
ax[1].set_title('Дней от даты счёта/УПД до оплаты'); ax[1].legend(fontsize=8.5, loc='lower right'); ax[1].grid(axis='y', visible=False)
plt.tight_layout()
show_fig(fig, 'B2B-компании зависят от нескольких клиентов; оплата по счетам приходит с отраслевой задержкой и длинным хвостом')

lost_disp = lost.sort_values('share', ascending=False).head(12).assign(share=lambda x: x.share.map(pct))
lost_disp = lost_disp.merge(co[['company_id', 'ind']], on='company_id')[['company_id', 'ind', 'cp_key', 'share', 'last_pay', 'co_last']]
show_table(lost_disp, 'Кандидаты в «уход ключевого клиента» (доля ≥15%, нет оплат >120 дней при живой компании)')
lag_disp = lag_ind.rename(columns={'count': 'оплат с датой документа', '50%': 'медиана, дн.', '90%': 'p90, дн.'}).round(0).astype(int).reset_index()
show_table(lag_disp, 'Задержка оплаты по счетам/УПД по отраслям')

finding(f'У {len(conc_b2b)} компаний с B2B-моделью (оплаты по счетам > 50% выручки) медианная доля крупнейшего клиента — {pct(conc_b2b.top1.median(),0)}, '
        f'медиана числа клиентов — {conc_b2b.clients.median():.0f}; у {(conc_b2b.top1 > 0.4).sum()} компаний крупнейший клиент даёт > 40% — '
        f'идеальные кандидаты для сценария «клиент заплатит на 2–4 недели позже». Найдено {len(lost)} случаев ухода ключевого клиента. '
        f'**Ключевая находка для прогноза:** в {pct(doc_date.notna().mean(),0)} B2B-оплат в назначении есть номер и дата счёта/УПД — по ним '
        f'можно восстановить дебиторку и распределение задержки оплаты (медиана {lag_ok.lag.median():.0f} дн., p90 {lag_ok.lag.quantile(.9):.0f} дн.), '
        f'{pct(partial,0)} оплат частичные/доплаты. Это прямой мост к задаче со звёздочкой «выставленные счета»: синтетический реестр счетов '
        f'можно сгенерировать из самой выписки, а затем честно измерить, насколько он улучшает прогноз.')

## 9. Граничные случаи

In [ ]:
section('9. Граничные случаи', 'Разовые крупные операции, временное закрытие, возвраты, взыскания')
grp = ops.groupby(['company_id', 'direction']).amount
ops['big'] = (ops.amount > 20 * grp.transform('median')) & (ops.amount > grp.transform(lambda s: s.quantile(0.999)))
big = ops[ops.big]

# временное закрытие: у «розничных» компаний (эквайринг/СБП/наличные > 50% выручки) ≥10 рабочих дней подряд без выручки.
# У B2B (стройка, агро) месяцы без оплат — норма сделочного бизнеса, а не закрытие, поэтому их не считаем.
retail_share = (ops[ops.category.isin(['Эквайринг (карты)', 'СБП (QR)', 'Наличная выручка'])].groupby('company_id').amount.sum()
                / ops[ops.is_revenue].groupby('company_id').amount.sum()).fillna(0)
rp = panel[panel.company_id.isin(retail_share[retail_share > 0.5].index)].copy()
rp['zero'] = rp.revenue <= 0
rp['run'] = (rp.zero != rp.groupby('company_id').zero.shift()).cumsum()
closed = (rp[rp.zero].groupby(['company_id', 'run'])
            .agg(ind=('ind', 'first'), начало=('date', 'min'), конец=('date', 'max'), раб_дней=('date', 'size'),
                 расходы_в_период=('outflow', 'sum'))
            .reset_index(level=1, drop=True).reset_index())
closed = closed[closed.раб_дней >= 10]
closed['что_происходило'] = np.where(closed.расходы_в_период > 0, 'точка закрыта, расходы идут', 'нет операций (пропуск выписки?)')
closed_m = closed

edge = pd.DataFrame([
    ('Разовые крупные операции (>20× медианы и >p99,9 компании)', len(big), big.company_id.nunique(), rub(big.amount.sum())),
    ('Розница: ≥10 раб. дней подряд без выручки (временное закрытие)', len(closed), closed.company_id.nunique(), '—'),
    ('Возвраты / ошибочные платежи', ops.category.eq('Возврат / ошибочный платёж').sum(), ops.loc[ops.category.eq('Возврат / ошибочный платёж'), 'company_id'].nunique(),
     rub(ops.loc[ops.category.eq('Возврат / ошибочный платёж'), 'amount'].sum())),
    ('Списания по исполнительному листу', ops.category.eq('Взыскания (исп. лист)').sum(), ops.loc[ops.category.eq('Взыскания (исп. лист)'), 'company_id'].nunique(),
     rub(ops.loc[ops.category.eq('Взыскания (исп. лист)'), 'amount'].sum())),
    ('Кредиты получены (разовые крупные поступления)', ops.category.eq('Кредит получен').sum(), ops.loc[ops.category.eq('Кредит получен'), 'company_id'].nunique(),
     rub(ops.loc[ops.category.eq('Кредит получен'), 'amount'].sum())),
    ('Перевод собственных средств в другой банк (уход денег из видимости)', (ops.category.eq('Перевод собственных средств') & ~ops.inflow).sum(),
     ops.loc[ops.category.eq('Перевод собственных средств') & ~ops.inflow, 'company_id'].nunique(),
     rub(ops.loc[ops.category.eq('Перевод собственных средств') & ~ops.inflow, 'amount'].sum())),
], columns=['случай', 'операций / месяцев', 'компаний', 'сумма'])
edge['операций / месяцев'] = edge['операций / месяцев'].map(num)
show_table(edge, 'Граничные случаи в данных')

big_cat = big.groupby(['direction', 'category']).size().sort_values(ascending=False).head(10).rename('операций').reset_index()
show_table(big_cat, 'Чем являются разовые крупные операции')
show_table(closed.sort_values('раб_дней', ascending=False).assign(расходы_в_период=lambda x: x.расходы_в_период.map(rub)),
           'Периоды временного закрытия розничных точек (≥10 раб. дней без выручки)')

finding(f'Граничные случаи, обещанные в README, действительно есть и их немного: {len(big)} разовых крупных операций у {big.company_id.nunique()} компаний '
        f'(в основном {", ".join(big_cat.category.head(3))}), {len(closed)} периодов временного закрытия розничных точек у {closed.company_id.nunique()} компаний '
        f'(≥10 раб. дней без выручки; у B2B «пустые» месяцы — норма сделочного бизнеса, а не закрытие), возвраты, исполнительные листы. Разовые крупные операции нельзя выдавать моделью «в среднем» — их надо '
        f'исключать из обучения статистической части (робастные оценки, медианы) и добавлять в прогноз только как известные события '
        f'(договор, кредит, ручной ввод). Переводы собственных средств в другой банк — не расход бизнеса, а перекладывание денег: '
        f'в прогнозе их лучше моделировать отдельно или исключать из «обязательных» платежей.')

## 10. Выводы и рекомендации для решения

In [ ]:
section('10. Выводы и рекомендации', 'Что из этого следует для категоризации, прогноза и проверки качества')
CONCL = f"""
**С какими данными мы имеем дело.** {co.company_id.nunique()} компаний, 15 отраслей, {num(len(ops))} операций за 2 года; один расчётный счёт
в одном банке. Данные синтетические, но генератор воспроизводит реальные механики: эквайринг T+1, ЕНП 28-го, зарплата в две волны,
оплаты по счетам с задержкой, сезонность, овердрафт и реакция владельца на нехватку денег. Данные чистые: остаток сходится с потоками
во всех днях, кроме дублей и «дыр» выгрузки, которые надёжно детектируются.

**Рекомендации.**
1. **Предобработка:** удалять дубли через сверку с остатком; помечать «дыры» (пропуск ≥5 раб. дней + скачок остатка) и не учить на них
   модель потоков; строить ряд на календаре рабочих дней РФ; ИНН — строкой.
2. **Категоризация:** правила + доразметка по истории контрагента уже дают ~99% покрытия — это минимальное требование кейса
   «закрыто дёшево и объяснимо». ML-слой — как сравнение подходов, а не как необходимость.
3. **Прогноз = скелет + статистика:** (а) детерминированные события — регулярные списания (аренда, кредит, лизинг, подписки),
   ЕНП по налоговому календарю, зарплата по двум датам; (б) выручка — дневная модель с эффектами дня недели (понедельник ×{r_wd.iloc[0]/r_wd.iloc[1:5].mean():.1f}),
   сезонностью (отраслевой профиль для коротких историй) и сдвигом эквайринга T+1; (в) B2B-поступления — из восстановленной дебиторки
   (номер и дата счёта в назначении) с эмпирическим распределением задержки. Вероятность разрыва — Монте-Карло по сценариям выручки и задержек.
4. **Определение разрыва:** прогнозировать остаток **без** будущих денег владельца и без переводов между своими счетами; разрыв —
   остаток ниже нуля (или ниже суммы ближайших обязательных платежей). Иначе модель «научится», что денег всегда хватает.
5. **Валидация:** rolling-origin backtest — несколько отсечек (например, каждые 2 недели в последние 6 месяцев), горизонт 30 раб. дней;
   метрики остатка — MAE/sMAPE в долях среднего дневного оборота, MASE против наивного прогноза «остаток не меняется» и «сезонно-наивного»;
   метрики разрывов — recall, precision (ложные тревоги), медиана заблаговременности в днях, ошибка суммы разрыва. Финальный hold-out —
   июль–сентябрь 2026 ({hold_any} компаний с разрывами).
6. **Бейзлайны для сравнения подходов:** (1) наивный остаток, (2) средний дневной поток за 90 дней, (3) скелет регулярных + сезонная выручка,
   (4) градиентный бустинг/Prophet по дневному потоку. Показывать, что каждый слой (календарь, счета, налоги) снижает ошибку — это напрямую
   отвечает на критерий «насколько доп. источники улучшают прогноз».
7. **Скрытый датасет жюри:** качество проверяют на отдельных данных — значит, никаких констант, «зашитых» под эти 150 компаний:
   правила категорий, детектор регулярности, сезонные профили и пороги разрыва выводятся из истории самой компании или берутся
   как отраслевые априоры; пайплайн обязан переживать короткую историю, пропуски, дубли и новые формулировки назначений,
   и иметь пакетный режим «папка выписок → прогнозы». Отложить часть компаний целиком (валидация по компаниям) — честная имитация такой проверки.
8. **Демо-сценарии:** готовые «реалистичные» кейсы из данных — компания с овердрафтом, компания с уходом ключевого клиента, сезонный бизнес
   на спаде (детский центр летом, курорт зимой), B2B с высокой концентрацией клиента (сценарий «клиент платит на 3 недели позже»).
"""
CONCL = ru_dec(CONCL)
display(Markdown(CONCL))
REPORT.append(('p', CONCL))

demo = pd.DataFrame([
    ('Овердрафт / явный разрыв', ', '.join(top_neg[:3])),
    ('Скрытый разрыв (владелец докладывает)', ', '.join(hid.company_id.value_counts().index[:3])),
    ('Уход ключевого клиента', ', '.join(lost.sort_values('share', ascending=False).company_id.drop_duplicates().head(3))),
    ('Высокая концентрация клиента (сценарий задержки)', ', '.join(conc_b2b.sort_values('top1', ascending=False).index[:3])),
    ('Временное закрытие точки', ', '.join(closed_m.company_id.drop_duplicates().head(3))),
    ('Короткая история (холодный старт)', ', '.join(short_hist.company_id.head(3))),
], columns=['сценарий для демо', 'компании-кандидаты'])
show_table(demo, 'Кандидаты для демонстрационных сценариев')

## 11. Экспорт HTML-отчёта
Собираем все графики, таблицы и выводы в один автономный HTML-файл (картинки встроены), который можно открыть в браузере или переслать команде.

In [ ]:
def md_to_html(md):
    out, in_list, in_ol = [], False, False
    for line in md.strip().split('\n'):
        s = line.strip()
        s_html = _html.escape(s)
        s_html = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', s_html)
        s_html = re.sub(r'`(.+?)`', r'<code>\1</code>', s_html)
        m = re.match(r'^(\d+)\.\s+(.*)', s_html)
        if m:
            if not in_ol: out.append('<ol>'); in_ol = True
            out.append(f'<li>{m.group(2)}'); continue
        if line.startswith('   ') and in_ol and s:
            out[-1] += ' ' + s_html; continue
        if in_ol and s and not m:
            out.append('</ol>'); in_ol = False
        if s:
            out.append(f'<p>{s_html}</p>')
    if in_ol: out.append('</ol>')
    return '\n'.join(out)

def build_report(path='cashflow_eda_report.html'):
    nav, body, sid = [], [], 0
    for item in REPORT:
        kind = item[0]
        if kind == 'kpi':
            tiles = ''.join(f'<div class="kpi"><div class="kpi-l">{_html.escape(k)}</div><div class="kpi-v">{_html.escape(v)}</div></div>' for k, v in item[1])
            body.append(f'<div class="kpis">{tiles}</div>')
        elif kind == 'h':
            sid += 1
            if sid > 1: body.append('</section>')
            nav.append(f'<a href="#s{sid}">{_html.escape(item[1])}</a>')
            body.append(f'<section id="s{sid}"><h2>{_html.escape(item[1])}</h2><p class="sub">{_html.escape(item[2])}</p>')
        elif kind == 'p':
            body.append(f'<div class="prose">{md_to_html(item[1])}</div>')
        elif kind == 'finding':
            body.append(f'<div class="finding"><div class="finding-t">Вывод</div>{md_to_html(item[1])}</div>')
        elif kind == 'fig':
            body.append(f'<figure><img src="data:image/png;base64,{item[2]}" alt="{_html.escape(item[1])}"><figcaption>{_html.escape(item[1])}</figcaption></figure>')
        elif kind == 'table':
            body.append(f'<div class="tblwrap"><div class="tbl-cap">{_html.escape(item[1])}</div>{item[2]}</div>')
    body.append('</section>')
    page = f"""<!doctype html>
<html lang="ru"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Cashflow EDA отчёт</title>
<style>
:root {{ --bg:#f9f9f7; --surface:#fcfcfb; --ink:#0b0b0b; --ink2:#52514e; --muted:#898781; --line:#e1e0d9; --accent:#2a78d6;
        --accent-wash:#eef4fc; --border:rgba(11,11,11,.10); color-scheme: light; }}
@media (prefers-color-scheme: dark) {{ :root:not([data-theme="light"]) {{ --bg:#0d0d0d; --surface:#1a1a19; --ink:#f0efec; --ink2:#c3c2b7;
        --muted:#898781; --line:#2c2c2a; --accent:#3987e5; --accent-wash:#16243a; --border:rgba(255,255,255,.10); color-scheme: dark; }} }}
:root[data-theme="dark"] {{ --bg:#0d0d0d; --surface:#1a1a19; --ink:#f0efec; --ink2:#c3c2b7; --muted:#898781; --line:#2c2c2a;
        --accent:#3987e5; --accent-wash:#16243a; --border:rgba(255,255,255,.10); color-scheme: dark; }}
* {{ box-sizing: border-box; }}
body {{ margin:0; background:var(--bg); color:var(--ink); font:15px/1.55 system-ui,-apple-system,"Segoe UI",sans-serif; }}
header {{ padding:40px 16px 24px; max-width:1180px; margin:0 auto; }}
header h1 {{ font-size:28px; line-height:1.2; margin:0 0 8px; letter-spacing:-.01em; }}
header p {{ margin:0; color:var(--ink2); max-width:820px; }}
nav {{ position:sticky; top:0; z-index:5; background:var(--bg); border-bottom:1px solid var(--line); }}
nav .in {{ max-width:1180px; margin:0 auto; padding:10px 16px; display:flex; gap:6px; overflow-x:auto; }}
nav a {{ white-space:nowrap; font-size:13px; color:var(--ink2); text-decoration:none; padding:4px 10px; border-radius:999px; }}
nav a:hover {{ background:var(--accent-wash); color:var(--ink); }}
main {{ max-width:1180px; margin:0 auto; padding:8px 16px 64px; }}
section {{ margin-top:40px; }}
h2 {{ font-size:22px; margin:0 0 2px; }}
.sub {{ color:var(--muted); margin:0 0 16px; font-size:14px; }}
.kpis {{ display:grid; grid-template-columns:repeat(auto-fill,minmax(200px,1fr)); gap:12px; margin:8px 0 8px; }}
.kpi {{ background:var(--surface); border:1px solid var(--border); border-radius:12px; padding:14px 16px; }}
.kpi-l {{ color:var(--muted); font-size:12px; text-transform:uppercase; letter-spacing:.04em; }}
.kpi-v {{ font-size:20px; font-weight:600; margin-top:4px; }}
.finding {{ background:var(--accent-wash); border-left:3px solid var(--accent); border-radius:8px; padding:12px 16px; margin:16px 0; }}
.finding-t {{ font-size:12px; font-weight:700; text-transform:uppercase; letter-spacing:.05em; color:var(--accent); margin-bottom:4px; }}
.finding p, .prose p {{ margin:6px 0; }}
.prose ol {{ padding-left:22px; }} .prose li {{ margin:6px 0; }}
code {{ font-family:ui-monospace,Consolas,monospace; font-size:.9em; background:var(--line); padding:1px 5px; border-radius:4px; }}
figure {{ margin:16px 0; background:#fcfcfb; border:1px solid var(--border); border-radius:12px; padding:12px; }}
figure img {{ width:100%; height:auto; display:block; }}
figcaption {{ color:#52514e; font-size:13px; margin-top:8px; }}
.tblwrap {{ margin:16px 0; overflow-x:auto; background:var(--surface); border:1px solid var(--border); border-radius:12px; padding:12px; }}
.tbl-cap {{ font-weight:600; font-size:14px; margin-bottom:8px; }}
table.tbl {{ border-collapse:collapse; width:100%; font-size:13px; font-variant-numeric:tabular-nums; }}
table.tbl th {{ text-align:left; color:var(--muted); font-weight:600; border-bottom:1px solid var(--line); padding:6px 10px; white-space:nowrap; }}
table.tbl td {{ border-bottom:1px solid var(--line); padding:6px 10px; vertical-align:top; }}
table.tbl tr:last-child td {{ border-bottom:none; }}
footer {{ color:var(--muted); font-size:12px; max-width:1180px; margin:0 auto; padding:0 16px 40px; }}
</style></head><body>
<header><h1>Банковские выписки МСП: разведочный анализ</h1>
<p>Кейс «Сервис финансового прогнозирования для малого бизнеса» (Центр-инвест, хакатон осень 2026). Что в данных, насколько им можно доверять,
какие закономерности заложены и что это значит для категоризации, прогноза остатка и поиска кассовых разрывов.</p></header>
<nav><div class="in">{''.join(nav)}</div></nav>
<main>{''.join(body)}</main>
<footer>Сгенерировано ноутбуком cashflow_eda.ipynb · данные синтетические, все компании и ИНН вымышленные.</footer>
</body></html>"""
    with open(path, 'w', encoding='utf-8') as f:
        f.write(page)
    print('Отчёт сохранён:', os.path.abspath(path), f'({os.path.getsize(path)/1e6:.1f} МБ)')
    return path

report_path = build_report()
if IN_COLAB:
    from google.colab import files
    files.download(report_path)